# Potenciais, Nernst e Tafel com os dados de uma célula real
**Tópicos Avançados em Ciências – Química Inorgânica Avançada · 2026.2 · Aula 01 (eletroquímica)**

Nesta atividade vamos:
1. construir o **diagrama de Pourbaix da água**, as duas retas de Nernst contra o pH;
2. calcular a **tensão reversível a 80 °C** e 0,53 bar, e reencontrar o 1,168 V do artigo;
3. ajustar a **reta de Tafel** do ânodo com dados medidos;
4. montar a **conta da tensão** da célula e ver onde o modelo da aula deixa de valer.

Os dados são da Fig. 7 de M. Bernt e H. A. Gasteiger, *J. Electrochem. Soc.* **163**, F3179 (2016), doi:10.1149/2.0231611jes: um eletrolisador PEM a 80 °C, eletrodo de IrO₂/TiO₂ com 11,6 % de ionômero. Os pontos foram digitalizados da figura, com 0,6 mV de resolução.

Complete as células marcadas com `# COMPLETE`. Atividade formativa: não vale nota.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

F = 96485.33212      # C/mol, a constante de Faraday: a carga de um mol de elétrons
R = 8.314462618      # J/(mol K)
T25 = 298.15         # K
T80 = 353.15         # K
LN10 = np.log(10)

## 1. O diagrama de Pourbaix da água

Cada eletrodo segue a equação de Nernst escrita como redução, $E = E^\circ - \frac{RT}{nF}\ln Q$, com $Q$ o quociente de reação. O H⁺ entra no $Q$ de cada eletrodo; com os gases a 1 bar:

$$E(\mathrm{O_2/H_2O}) = 1{,}229\ \mathrm{V} - \frac{RT}{F}\ln 10\cdot\mathrm{pH}, \qquad E(\mathrm{H^+/H_2}) = -\frac{RT}{F}\ln 10\cdot\mathrm{pH}.$$

In [ ]:
def E_O2(pH, T=T25):
    '''Potencial do par O2/H2O contra o EPH, em V, com O2 a 1 bar.'''
    return ...   # COMPLETE

def E_H2(pH, T=T25):
    '''Potencial do par H+/H2 contra o EPH, em V, com H2 a 1 bar.'''
    return ...   # COMPLETE

print(f"inclinação a 25 °C: {-(R*T25/F)*LN10*1e3:.2f} mV por unidade de pH")

In [ ]:
pH = np.linspace(0, 14, 50)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(pH, E_H2(pH), E_O2(pH), color="#1D4ED8", alpha=0.10)
ax.plot(pH, E_O2(pH), color="#B91C1C", lw=2.2, label="O$_2$/H$_2$O")
ax.plot(pH, E_H2(pH), color="#B91C1C", lw=2.2, ls="--", label="H$^+$/H$_2$")
ax.text(5, 0.3, "a água é estável", ha="center", weight="bold")
ax.set_xlabel("pH"); ax.set_ylabel("E contra o EPH (V)")
ax.set_xlim(0, 14); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
# a distância entre as retas depende do pH?
for p in (0, 7, 14):
    print(f"pH {p:2d}: E(O2) - E(H2) = {E_O2(p) - E_H2(p):.3f} V")

**Note:** a célula PEM trabalha em meio ácido, perto de pH 0, e a alcalina perto de pH 14. As duas têm a mesma tensão reversível: o H⁺ se cancela na soma das semirreações.

## 2. A tensão reversível a 80 °C e 0,53 bar

A tensão padrão varia com a entropia de reação, $\frac{dU^\circ}{dT} = -\frac{\Delta S^\circ}{nF}$, e Nernst corrige para as pressões:
$$U_{\mathrm{rev}} = U^\circ(T) + \frac{RT}{2F}\ln\left(p_{\mathrm{H_2}}\,p_{\mathrm{O_2}}^{1/2}\right).$$

A 80 °C e 1 bar total o vapor d'água ocupa 0,47 bar, e os gases ficam a $p_{\mathrm{H_2}} = p_{\mathrm{O_2}} = 0{,}53$ bar. Entropias padrão, em J/(mol K): H₂O(l) 69,95; H₂(g) 130,68; O₂(g) 205,15.

In [ ]:
U0_25 = 237.13e3 / (2 * F)                # V, de ΔG° = 237,13 kJ/mol
dS = ...        # COMPLETE: ΔS° da decomposição H2O(l) -> H2 + 1/2 O2
U0_80 = ...     # COMPLETE: U° a 80 °C
p = 0.53
U_rev = ...     # COMPLETE: Nernst com as pressões parciais
print(f"ΔS° = {dS:.1f} J/(mol K);  dU°/dT = {-dS/(2*F)*1e3:.3f} mV/K")
print(f"U°(80 °C) = {U0_80:.4f} V;  U_rev(80 °C; 0,53 bar) = {U_rev:.4f} V   (artigo: 1,168 V)")

## 3. A reta de Tafel do ânodo

A tensão livre da queda ôhmica ("iR-free") é $U_{\mathrm{rev}} + \eta_{\mathrm{OER}} + \eta_{\mathrm{HER}}$, e $\eta_{\mathrm{HER}}$ é desprezível no Pt. Entre 10 e 100 mA/cm² o artigo ajusta uma reta de Tafel:
$$\eta = b\,\log_{10}\frac{j}{j_0}.$$

In [ ]:
dados = pd.read_csv("dados/bernt2016_fig7_11p6wt.csv", comment="#")
j = dados["j_mA_cm2"].to_numpy() / 1000     # A/cm2
U_irfree = dados["U_iRfree_V"].to_numpy()   # V

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.semilogx(j * 1000, U_irfree, "o", mfc="none", color="#1D4ED8", ms=8,
            label="Fig. 7, 11,6 % de ionômero")
ax.set_xlabel("j (mA/cm$^2$)"); ax.set_ylabel("tensão livre de iR (V)")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.show()

In [ ]:
m = j <= 0.100                               # a faixa de ajuste do artigo
b, a = ...      # COMPLETE: ajuste linear de U_irfree contra log10(j), só em m
j_145 = ...     # COMPLETE: corrente em que a reta passa por 1,45 V
j0 = ...        # COMPLETE: corrente de troca (onde a reta cruza U_rev)
alfa = ...      # COMPLETE: alfa aparente, de b = 2,303 RT/(alfa F)
print(f"b = {b*1e3:.1f} mV por fator 10 na corrente   (Tabela I do artigo: 46)")
print(f"j em 1,45 V = {j_145*1e3:.1f} mA/cm2     (Tabela I: 90,1 mA/cm2)")
print(f"j0 = {j0:.1e} A/cm2;  alfa aparente = {alfa:.2f}")

A diferença para a Tabela I é da ordem da precisão da leitura da figura. Um passo simples de um elétron, com $\alpha = 0{,}5$, daria $b = 2{,}303\,RT/(0{,}5\,F) \approx 140$ mV por fator 10 na corrente, a 80 °C.

## 4. A conta da tensão, e onde ela falha

Modelo da aula: $U = U_{\mathrm{rev}} + b\log_{10}(j/j_0) + j\cdot\mathrm{HFR} + j\cdot R_{\mathrm{ct,HER}}$, com HFR = 52,5 mΩ cm² (medida por impedância) e $R_{\mathrm{ct,HER}} = 0{,}29$ mΩ cm². A tensão medida é a livre de iR mais $j\cdot\mathrm{HFR}$.

In [ ]:
HFR = 52.5e-3       # Ohm cm2
R_HER = 0.29e-3     # Ohm cm2
U_medida = U_irfree + j * HFR
U_modelo = ...      # COMPLETE
residuo_mV = (U_medida - U_modelo) * 1e3

k = int(np.argmin(abs(j - 1.0)))
print(f"a 1 A/cm2: modelo {U_modelo[k]:.3f} V, medida {U_medida[k]:.3f} V, "
      f"diferença {residuo_mV[k]:.0f} mV")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(j, U_medida, "o", mfc="none", color="#1D4ED8", label="medida")
ax[0].plot(j, U_modelo, "-", color="#B91C1C", label="modelo da aula")
ax[0].set_xlabel("j (A/cm$^2$)"); ax[0].set_ylabel("U (V)"); ax[0].legend()
ax[1].plot(j, residuo_mV, "o-", color="#B91C1C")
ax[1].set_xlabel("j (A/cm$^2$)"); ax[1].set_ylabel("medida − modelo (mV)")
for a_ in ax: a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Para discutir

1. O resíduo cresce com a corrente. Que processo cresce com $j$ e não está na reta de Tafel? O artigo atribui de 5 a 20 mV a ele em 3 A/cm². É por onde a Aula 02 começa.
2. Refaça o ajuste com os pontos de 10 a 300 mA/cm². O que acontece com $b$, e por que a janela de ajuste importa?
3. A célula PEM trabalha em meio ácido, perto de pH 0 no diagrama de Pourbaix. Por que o ânodo é de IrO₂, e não de um óxido de níquel, comum nos eletrolisadores alcalinos?
4. Com $\alpha$ aparente perto de 1,5, o que se pode dizer sobre as etapas da OER antes da etapa determinante?